# Лабораторная работа: ISBI-2012 сегментация

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import files
from PIL import Image
from pathlib import Path
import io,zipfile,tempfile,os
u=files.upload();name=next(n for n in u if n.endswith('.zip'))
root=tempfile.mkdtemp()
with zipfile.ZipFile(io.BytesIO(u[name])) as z:
 for member in z.infolist():
  dest=os.path.realpath(os.path.join(root,member.filename))
  assert dest.startswith(os.path.realpath(root)+os.sep) or dest==os.path.realpath(root)
 z.extractall(root)
imgs={p.stem:p for p in Path(root).rglob('*') if 'images' in p.parts and p.suffix.lower() in ('.png','.tif','.tiff')}
masks={p.stem:p for p in Path(root).rglob('*') if 'masks' in p.parts and p.suffix.lower() in ('.png','.tif','.tiff')}
keys=sorted(imgs.keys()&masks.keys());assert len(keys)>2,'Нужен ZIP с images/ и masks/'
X=np.array([np.asarray(Image.open(imgs[k]).convert('L').resize((128,128)))/255 for k in keys],dtype='float32')[...,None]
Y=np.array([np.asarray(Image.open(masks[k]).convert('L').resize((128,128),Image.Resampling.NEAREST))>127 for k in keys],dtype='float32')[...,None]
from sklearn.model_selection import train_test_split
a,b,c,d=train_test_split(X,Y,test_size=.2,random_state=42)
inp=keras.Input((128,128,1));x=layers.Conv2D(16,3,activation='relu',padding='same')(inp)
x=layers.MaxPooling2D()(x);x=layers.Conv2D(32,3,activation='relu',padding='same')(x)
x=layers.MaxPooling2D()(x);x=layers.Conv2D(64,3,activation='relu',padding='same')(x)
x=layers.UpSampling2D()(x);x=layers.Conv2D(32,3,activation='relu',padding='same')(x)
x=layers.UpSampling2D()(x);out=layers.Conv2D(1,1,activation='sigmoid')(x)
model=keras.Model(inp,out);model.compile('adam','binary_crossentropy',metrics=['accuracy',keras.metrics.BinaryIoU(target_class_ids=[1],threshold=.5)])
history=model.fit(a,c,validation_data=(b,d),epochs=20,batch_size=4)
pd.DataFrame(history.history).plot(grid=True);plt.show()